In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
         
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ml-contest-cuet/sample_submission.csv
/kaggle/input/ml-contest-cuet/train.csv
/kaggle/input/ml-contest-cuet/test.csv
/kaggle/input/test-data-augmentation/test_data_translated_test.csv.csv


In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth 
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

In [3]:
from unsloth import FastLanguageModel
from unsloth import FastModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)  

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-07-02 14:49:55.624278: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751467795.828027      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751467795.886297      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.12: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(  
    model, 
    r = 16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=32, 
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
) 

Unsloth 2025.6.12 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [5]:
import pandas as pd
from tqdm import tqdm
df_test= pd.read_csv("/kaggle/input/test-data-augmentation/test_data_translated_test.csv.csv")

In [6]:
# df_test
df_test['translated']

0      1. Question: The magnetic susceptibility of a ...
1      2. Question: A bullet loses half of its veloci...
2      3. Question: If the length of a simple pendulu...
3      4. Question: A rocket loses 1/60160 of its mas...
4      5. Question: A circular coil has 50 turns and ...
                             ...                        
195    6. Question: A bulb is marked 100W-200V. What ...
196    7. Question: What is the property by which it ...
197    8. Question: The distance between the two lens...
198    9. Question: The cross-sectional area of a wir...
199    10. Question: Electricity flows through the th...
Name: translated, Length: 200, dtype: object

In [7]:
from unsloth.chat_templates import get_chat_template
import pandas as pd
import torch

# prompt_template = """
# পদার্থবিজ্ঞানের MCQ বিশেষজ্ঞ হিসাবে, নিচের নিয়মগুলি অনুসরণ করুন:
# 1. শুধুমাত্র সঠিক উত্তরের অক্ষর দিন (A, B, C, বা D)
# 2. কোনো ব্যাখ্যা দেবেন না
# 3. পদার্থবিজ্ঞানের সূত্র ব্যবহার করুন (যেমন: \(F = ma\), \(K.E. = \frac{1}{2}mv^2\))
# 4. উত্তর শুধুমাত্র একটি অক্ষর হবে

# উদাহরণ:
# প্রশ্ন: 10 m/s বেগে চলমান 2 kg ভরের বস্তুর গতিশক্তি কত?
# A. 50 J B. 100 J C. 150 J D. 200 J
# উত্তর: B

# নতুন প্রশ্ন:
# {question}

# উত্তর:
# """ 

prompt_template = """
You are a Physics MCQ expert at solving HSC-level Physics multiple-choice questions (MCQs). For each question, you will be given four multiple-choice options. Your task is to reply ONLY with the correct option's letter (A, B, C, or D) on a new line.

RULES:
1. The questions are primarily from HSC-level Physics, but sometimes may be from other subjects.
2. If the question is outside Physics, you must still select the most appropriate answer from the options.
3. Do NOT give any explanation or reasoning.
4. ONLY reply with a single letter: A, B, C, or D.
5. The letter must be on a new line.
6. Do NOT show your calculations or thinking process.
7. Always choose exactly one answer.


# Example 1:
# Question: ...
# Options: ...
# Answer: B

# Example 2:
# Question: ...
# Options: ...
# Answer: D


Now answer this question:
{question}

Answer:
"""



# prompt_template = """You are a Physics MCQ expert. For each question with four multiple-choice options, reply only with the correct option's letter (A, B, C, or D).
# Always choose one and only one answer.

# RULES:
# 1. DO NOT give any explanation.
# 2. ONLY reply with a single letter: A, B, C, or D.
# 3. The letter must be on a new line.
# 4. DO NOT show your thinking process.

# Example:
# Question: The kinetic energy of a body with a mass of 50 kg and momentum of 200 kgms⁻¹ is:
# Options: A. 200 J B. 300 J C. 400 J D. 500 J

# Answer:
# C

# Example 1:
# Question: ...
# Options: ...
# Answer: B

# Example 2:
# Question: ...
# Options: ...
# Answer: D

# Now answer this question:
# {question}

# Answer:
# """ 

responses = []

model = model.to("cuda")
tokenizer = tokenizer  # Assuming tokenizer is already loaded

for text in tqdm(df_test['translated']):
    prompt = prompt_template.format(question=text)
    messages = [{"role": "user", "content": prompt}]
    # messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}] ## used for gemma 12B

    # This returns a tensor (input_ids) directly
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        padding=True,  # Enable padding
        # enable_thinking = False,
        return_dict=True  # Return input_ids and attention_mask
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],  # Pass attention mask
            max_new_tokens=2048,
            # temperature=0.1,
            # top_p=0.95,
            # top_k=50,
            temperature = 0.2,
            do_sample=False,
            top_p = 0.95,
            top_k = 10, # For thinking
            use_cache=True,
            
        )

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract the assistant response between the special tokens
    start_token = "<|im_start|>assistant"
    end_token = "<|im_end|>"

    start_idx = decoded_output.find(start_token)
    end_idx = decoded_output.find(end_token, start_idx + len(start_token))

    if start_idx != -1 and end_idx != -1:
        response = decoded_output[start_idx + len(start_token):end_idx].strip()
    else:
        response = decoded_output.strip()

    # print(f"This is the response {response}")
    responses.append(response)
    

100%|██████████| 200/200 [2:16:19<00:00, 40.90s/it]  


In [8]:
import re
 
cleaned_responses = [] 

for res in responses:
    match = re.search(r'</think>\s*([A-D])', res)
    if match:
        cleaned_responses.append(match.group(1))
    else:
        cleaned_responses.append("")

print(cleaned_responses)  

['C', 'A', 'D', '', 'B', 'B', 'C', 'C', 'B', 'B', 'B', 'B', 'C', 'D', 'A', 'B', 'C', 'A', 'C', 'C', 'C', 'B', 'B', 'B', 'D', 'C', 'A', 'B', 'A', 'B', 'A', 'D', 'B', 'B', 'B', 'A', 'C', 'A', '', 'B', 'D', 'B', 'D', 'A', 'B', 'A', 'C', 'C', 'D', 'A', 'A', 'B', 'D', 'B', 'D', 'D', 'B', 'D', 'C', 'C', 'C', 'C', 'B', 'B', 'B', 'B', 'D', 'B', 'C', 'B', 'C', 'B', 'C', '', 'D', 'B', 'C', 'A', 'B', 'B', 'C', 'A', 'C', 'C', 'A', 'D', 'B', 'B', 'A', 'B', 'B', 'C', 'B', 'D', 'D', 'B', 'D', 'D', 'B', 'A', 'B', 'C', 'B', 'C', '', 'A', 'A', 'B', 'A', 'A', 'B', 'B', 'A', 'D', 'A', 'C', 'B', 'A', 'D', 'D', 'C', 'C', 'B', 'C', '', 'D', 'D', 'B', 'B', 'C', 'A', 'B', 'A', 'B', 'B', 'B', 'B', 'A', 'D', 'C', 'C', 'C', 'A', 'A', 'D', 'B', 'B', 'C', 'A', 'A', 'B', '', 'C', 'B', 'C', 'B', 'A', 'A', 'B', 'D', 'A', 'A', 'B', 'A', 'C', 'B', 'A', '', 'B', 'A', 'D', 'A', 'B', 'C', 'B', 'A', 'D', 'B', 'B', 'A', 'D', 'C', 'B', 'B', 'A', 'C', 'A', 'A', 'B', 'B', 'A', 'B', 'D', 'D', 'A', 'C', 'D', 'B', '', 'A']


In [9]:
df_test['answer']=cleaned_responses
df_test

,Unnamed: 0,id,question,options,formatted,translated,answer
0,0,1,কোনো বস্তুর চৌম্বকত্ব ধারকত্ব পরিমাপ করা হয়-,['চুম্বকনকারি বলা হয় ' 'সম্পৃক্ত দ্বারা ' 'আবি...,Question: কোনো বস্তুর চৌম্বকত্ব ধারকত্ব পরিমাপ...,1. Question: The magnetic susceptibility of a ...,C
1,1,2,একটি বুলেট লক্ষ বস্তুর 3 cm ভেতরে প্রবেশ করতে ...,['1cm' '1m' '2 cm' '2 m'],Question: একটি বুলেট লক্ষ বস্তুর 3 cm ভেতরে প্...,2. Question: A bullet loses half of its veloci...,A
2,2,3,"একটি সরল দোলকের দোলনকাল T, দৈর্ঘ্য দ্বিগুণ হলে...",['2T' '`T/sqrt2`' '`T/2`' '`sqrt2T`'],"Question: একটি সরল দোলকের দোলনকাল T, দৈর্ঘ্য দ...",3. Question: If the length of a simple pendulu...,D
3,3,4,একটি রকেট ঊর্ধমুখী যাত্রায় প্রথম 2 সেকেন্ড এর ...,['36 ms-2' '20.2 ms-2' '15.2 ms-2' '30 ms-2'],Question: একটি রকেট ঊর্ধমুখী যাত্রায় প্রথম 2 স...,4. Question: A rocket loses 1/60160 of its mas...,
4,4,5,50 পাকের একটি বৃত্তাকার কুন্ডলীর ব্যাস 30 cm। ...,['4.7 × 10-5 A' '0.48 A' '47 A' '94 A'],Question: 50 পাকের একটি বৃত্তাকার কুন্ডলীর ব্য...,5. Question: A circular coil has 50 turns and ...,B
...,...,...,...,...,...,...,...
195,195,196,একটি বালের গায়ে 100W-200V লেখা। এর রোধ কত?,['200Ω' '300Ω' '400Ω' '500Ω'],Question: একটি বালের গায়ে 100W-200V লেখা। এর ...,6. Question: A bulb is marked 100W-200V. What ...,C
196,196,197,যে বৈশিষ্ট্য দ্বারা একটি শব্দ অন্য একটি শব্দ হ...,['তীব্রতা' 'স্বরকম্প' 'স্বরগ্রাম' 'শব্দোচ্চতা'],Question: যে বৈশিষ্ট্য দ্বারা একটি শব্দ অন্য এ...,7. Question: What is the property by which it ...,D
197,197,198,16 বিবর্ধন বিশিষ্ট নভো-দূরবীক্ষন যন্তের লেন্সে...,"['100 cm, 10 cm' '80 cm, 5 cm' '200 cm, 2 cm' ...",Question: 16 বিবর্ধন বিশিষ্ট নভো-দূরবীক্ষন যন্...,8. Question: The distance between the two lens...,B
198,198,199,একটি তারের প্রস্থচ্ছেদ এর ক্ষেত্রফল 0.003 m2 ।...,['102kg' '102 নিউটন ' '9.8 ×102kg' '9.8×102 নি...,Question: একটি তারের প্রস্থচ্ছেদ এর ক্ষেত্রফল ...,9. Question: The cross-sectional area of a wir...,


In [10]:
df_test = df_test[['id', 'answer']]
df_test

,id,answer
0,1,C
1,2,A
2,3,D
3,4,
4,5,B
...,...,...
195,196,C
196,197,D
197,198,B
198,199,


In [11]:
df_test.to_csv('translated_submission.csv', index=False) 